In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

# === CONFIGURACIÓN ===
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
BASE_OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
MODEL_NAME = "Logistic"
INTENTO = 6  # Cambiar manualmente si es otro intento

# Crear subcarpeta de salida
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f"{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# === ENTRENAMIENTO ===
metricas_totales = []
mejor_modelo = None
mejor_score = -1

# Detectar variantes
variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith("X_train")])

for x_file in variantes_X:
    base_name = x_file.replace("X_train_", "").replace(".parquet", "")
    try:
        print(f"🔍 Procesando: {base_name}")

        # Cargar archivos
        X_train = pd.read_parquet(os.path.join(INPUT_PATH, f"X_train_{base_name}.parquet"))
        X_test = pd.read_parquet(os.path.join(INPUT_PATH, f"X_test_{base_name}.parquet"))
        y_train = pd.read_parquet(os.path.join(INPUT_PATH, f"y_train_{base_name}.parquet")).squeeze()
        y_test = pd.read_parquet(os.path.join(INPUT_PATH, f"y_test_{base_name}.parquet")).squeeze()

        # Entrenamiento
        start = time.time()
        model = LogisticRegression(max_iter=500, multi_class='multinomial', solver='lbfgs', class_weight='balanced')
        model.fit(X_train, y_train)
        tiempo = (time.time() - start) / 60

        # Predicción
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        # Reporte completo
        report_dict = classification_report(y_test, y_pred_test, output_dict=True)
        df_report = pd.DataFrame(report_dict).transpose()
        df_report["tiempo_min"] = tiempo

        # Métricas adicionales
        f1_1 = df_report.loc['1', 'f1-score'] if '1' in df_report.index else 0
        recall_1 = df_report.loc['1', 'recall'] if '1' in df_report.index else 0

        if f1_1 > mejor_score:
            mejor_score = f1_1
            mejor_modelo = base_name

        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred_test)

        # Guardar CSV de métricas
        df_report.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{base_name}.csv"))

        # Guardar PDF con visualizaciones
        with PdfPages(os.path.join(OUTPUT_PATH, f"reporte_{base_name}.pdf")) as pdf:
            # Confusion Matrix
            ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
            plt.title("Matriz de Confusión")
            pdf.savefig()
            plt.close()

            # ROC por clase
            classes = np.unique(y_test)
            y_bin = label_binarize(y_test, classes=classes)
            plt.figure()
            for i, clase in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={roc_auc:.2f})")
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title("Curvas ROC por clase")
            plt.xlabel("FPR")
            plt.ylabel("TPR")
            plt.legend()
            pdf.savefig()
            plt.close()

            # PR Curve por clase
            plt.figure()
            for i, clase in enumerate(classes):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
                pr_auc = average_precision_score(y_bin[:, i], y_proba[:, i])
                plt.plot(recall, precision, label=f"Clase {clase} (PR AUC={pr_auc:.2f})")
            plt.title("Curvas Precision-Recall por clase")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.legend()
            pdf.savefig()
            plt.close()

        # Resumen final
        metricas_totales.append({
            "modelo": base_name,
            "f1_nivel_1": f1_1,
            "recall_nivel_1": recall_1,
            "tiempo_min": tiempo,
            "accuracy": accuracy_score(y_test, y_pred_test),
            "macro_f1": df_report.loc['macro avg', 'f1-score'],
            "weighted_f1": df_report.loc['weighted avg', 'f1-score'],
            "train_accuracy": accuracy_score(y_train, y_pred_train),
            "test_accuracy": accuracy_score(y_test, y_pred_test)
        })

        print(f"✅ {base_name}: F1_clase_1={f1_1:.3f}, Recall_clase_1={recall_1:.3f}, Tiempo={tiempo:.2f}min")

    except Exception as e:
        print(f"❌ Error en {base_name}: {e}")

# Guardar resumen final
pd.DataFrame(metricas_totales).to_csv(
    os.path.join(OUTPUT_PATH, "resumen_modelos_logistic_regression.csv"), index=False)

print(f"\n🏆 Mejor modelo según F1 clase 1: {mejor_modelo} (F1 = {mejor_score:.4f})")
